In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as mpatheffects
from datetime import timedelta
import pandas as pd
import os
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from scipy.interpolate import RegularGridInterpolator, make_interp_spline
import matplotlib.patches as mpatches
import geopandas as gpd
from matplotlib.lines import Line2D
from shapely.ops import unary_union

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.unicode_minus"] = False

# ================== 输出目录 ==================
output_dir = r"E:\关中干旱小论文\研究区域图-图一"
os.makedirs(output_dir, exist_ok=True)

# ================== 时间设置 ==================
start_date = pd.to_datetime("2025-03-01")
end_date   = pd.to_datetime("2025-05-31")

LON_MIN, LON_MAX = 60, 140
LAT_MIN, LAT_MAX = 0,  50
LEVEL_USE = 850

# ================== 时间分段 ==================
date_ranges = []
cur = start_date
while cur <= end_date:
    seg_end = min(cur + timedelta(days=14), end_date)
    date_ranges.append((cur, seg_end))
    cur += timedelta(days=15)
date_ranges = date_ranges[:6]

# ================== 坐标刻度 ==================
lon_ticks = [60, 80, 100, 120, 140]
lat_ticks = [0, 10, 20, 30, 40, 50]

# ================== 箭头参数 ==================
QUIVER_STRIDE  = 1
WIND_THRESHOLD = 1.5
QUIVER_SCALE   = 40
QUIVER_KEY_U   = 4

# ================== 流线追踪参数 ==================
BOB_SEEDS   = [(85, 12), (88, 14)]
SCS_SEEDS   = [(112, 8), (114, 10)]
TRACE_DT    = 0.4
TRACE_STEPS = 120

# =========================================================
# 预处理 Shapefile：dissolve 合并 → 单一轮廓几何体
# 只执行一次，所有子图共用
# =========================================================
GZ_SHP_PATH = r"E:\矢量边界\gz_bountary\gz_city.shp"
_gz_raw = gpd.read_file(GZ_SHP_PATH)

# 统一投影到 WGS84
if _gz_raw.crs is not None and _gz_raw.crs.to_epsg() != 4326:
    _gz_raw = _gz_raw.to_crs(epsg=4326)

# ✅ 关键：合并所有要素为单一几何，消除内部行政边界
GZ_UNION = unary_union(_gz_raw.geometry.values)

# =========================================================
# 插值函数
# =========================================================
def get_uv_from_interp(u_interp, v_interp, lon, lat,
                       lon_range=(LON_MIN, LON_MAX),
                       lat_range=(LAT_MIN, LAT_MAX)):
    if not (lon_range[0] <= lon <= lon_range[1] and
            lat_range[0] <= lat <= lat_range[1]):
        return 0.0, 0.0
    try:
        u_val = u_interp([[lat, lon]])[0]
        v_val = v_interp([[lat, lon]])[0]
    except Exception:
        return 0.0, 0.0
    return float(u_val), float(v_val)


# =========================================================
# RK4 流线追踪
# =========================================================
def rk4_trace(u_interp, v_interp, lon0, lat0,
              dt=TRACE_DT, steps=TRACE_STEPS):
    path_lon = [float(lon0)]
    path_lat = [float(lat0)]
    lon = float(lon0)
    lat = float(lat0)

    for _ in range(steps):
        u1, v1 = get_uv_from_interp(u_interp, v_interp, lon, lat)
        if np.sqrt(u1**2 + v1**2) < 0.3:
            break
        cos1  = max(np.cos(np.radians(lat)), 0.01)
        dlon1 = (u1 / (111000.0 * cos1)) * (dt * 111000.0)
        dlat1 = (v1 / 111000.0) * (dt * 111000.0)

        u2, v2 = get_uv_from_interp(u_interp, v_interp,
                                     lon + dlon1 / 2.0,
                                     lat + dlat1 / 2.0)
        cos2  = max(np.cos(np.radians(lat + dlat1 / 2.0)), 0.01)
        dlon2 = (u2 / (111000.0 * cos2)) * (dt * 111000.0)
        dlat2 = (v2 / 111000.0) * (dt * 111000.0)

        u3, v3 = get_uv_from_interp(u_interp, v_interp,
                                     lon + dlon2 / 2.0,
                                     lat + dlat2 / 2.0)
        cos3  = max(np.cos(np.radians(lat + dlat2 / 2.0)), 0.01)
        dlon3 = (u3 / (111000.0 * cos3)) * (dt * 111000.0)
        dlat3 = (v3 / 111000.0) * (dt * 111000.0)

        u4, v4 = get_uv_from_interp(u_interp, v_interp,
                                     lon + dlon3,
                                     lat + dlat3)
        cos4  = max(np.cos(np.radians(lat + dlat3)), 0.01)
        dlon4 = (u4 / (111000.0 * cos4)) * (dt * 111000.0)
        dlat4 = (v4 / 111000.0) * (dt * 111000.0)

        lon += (dlon1 + 2.0*dlon2 + 2.0*dlon3 + dlon4) / 6.0
        lat += (dlat1 + 2.0*dlat2 + 2.0*dlat3 + dlat4) / 6.0

        if not (LON_MIN <= lon <= LON_MAX and LAT_MIN <= lat <= LAT_MAX):
            break

        path_lon.append(lon)
        path_lat.append(lat)

    return np.array(path_lon), np.array(path_lat)


def trace_branch(u_interp, v_interp, seeds):
    valid_paths = []
    for lon0, lat0 in seeds:
        lons, lats = rk4_trace(u_interp, v_interp, lon0, lat0)
        if len(lons) >= 5:
            valid_paths.append((lons, lats))
    if not valid_paths:
        return None, None
    longest = max(valid_paths, key=lambda p: len(p[0]))
    return longest[0], longest[1]


def smooth_path_arr(lons, lats, n=300):
    if len(lons) < 4:
        return lons, lats
    t     = np.linspace(0, 1, len(lons))
    t_new = np.linspace(0, 1, n)
    try:
        spl_lon = make_interp_spline(t, lons, k=3)(t_new)
        spl_lat = make_interp_spline(t, lats, k=3)(t_new)
    except Exception:
        return lons, lats
    return spl_lon, spl_lat


# ================== 数据读取 ==================
def load_wind_period(uwnd_ds, vwnd_ds, start, end):
    u = uwnd_ds["uwnd"].sel(
        level=LEVEL_USE,
        lat=slice(LAT_MAX, LAT_MIN),
        lon=slice(LON_MIN, LON_MAX),
        time=slice(str(start.date()), str(end.date()))
    ).mean("time").compute()

    v = vwnd_ds["vwnd"].sel(
        level=LEVEL_USE,
        lat=slice(LAT_MAX, LAT_MIN),
        lon=slice(LON_MIN, LON_MAX),
        time=slice(str(start.date()), str(end.date()))
    ).mean("time").compute()

    return u, v


def build_interpolators(u_da, v_da):
    u_s  = u_da.sortby("lat")
    v_s  = v_da.sortby("lat")
    lats = u_s.lat.values.astype(float)
    lons = u_s.lon.values.astype(float)

    u_interp = RegularGridInterpolator(
        (lats, lons), u_s.values.astype(float),
        method="linear", bounds_error=False, fill_value=0.0
    )
    v_interp = RegularGridInterpolator(
        (lats, lons), v_s.values.astype(float),
        method="linear", bounds_error=False, fill_value=0.0
    )
    return u_interp, v_interp, u_s, v_s


# =========================================================
# 关中轮廓绘制
# ✅ 用 add_geometries() 而非 gdf.plot()，不破坏 ax 的 aspect
# =========================================================
def draw_guanzhong_boundary(ax, proj=None):
    if proj is None:
        proj = ccrs.PlateCarree()

    # 填充色块
    ax.add_geometries(
        [GZ_UNION],
        crs=proj,
        facecolor="#ffe0a0",
        edgecolor="#8B0000",
        linewidth=1.2,
        alpha=0.60,
        zorder=50
    )


# ================== 路径绘制 ==================
def draw_traced_branch(ax, path_lons, path_lats,
                       color, lw, label, n_arrows=5, proj=None):
    if proj is None:
        proj = ccrs.PlateCarree()
    if path_lons is None or len(path_lons) < 5:
        return

    spl_lon, spl_lat = smooth_path_arr(path_lons, path_lats, n=400)

    mask = (
        (spl_lon >= LON_MIN) & (spl_lon <= LON_MAX) &
        (spl_lat >= LAT_MIN) & (spl_lat <= LAT_MAX)
    )
    spl_lon = spl_lon[mask]
    spl_lat = spl_lat[mask]
    if len(spl_lon) < 8:
        return

    ax.plot(
        spl_lon, spl_lat,
        color=color, lw=lw, ls="-",
        transform=proj, zorder=60,
        path_effects=[
            mpatheffects.Stroke(linewidth=lw + 1.6,
                                foreground="white", alpha=0.80),
            mpatheffects.Normal()
        ]
    )

    idx_s      = int(len(spl_lon) * 0.08)
    idx_e      = int(len(spl_lon) * 0.92)
    arrow_idxs = np.linspace(idx_s, idx_e, n_arrows, dtype=int)
    trans      = proj._as_mpl_transform(ax)

    for idx in arrow_idxs:
        if idx + 8 >= len(spl_lon):
            continue
        x0, y0 = spl_lon[idx],     spl_lat[idx]
        x1, y1 = spl_lon[idx + 8], spl_lat[idx + 8]
        ax.annotate(
            "",
            xy=(x1, y1), xytext=(x0, y0),
            xycoords=trans, textcoords=trans,
            arrowprops=dict(
                arrowstyle="-|>",
                color=color,
                lw=lw * 0.9,
                mutation_scale=10,
            ),
            zorder=62
        )

    mid = len(spl_lon) // 2
    ax.text(
        spl_lon[mid], spl_lat[mid] + 1.6,
        label,
        fontsize=5.5, color=color, fontweight="bold",
        transform=proj, zorder=65, ha="center",
        bbox=dict(boxstyle="round,pad=0.15", fc="white",
                  ec=color, alpha=0.82, lw=0.5)
    )


# ================== 边框 ==================
def draw_domain_border(ax):
    xs = [LON_MIN, LON_MAX, LON_MAX, LON_MIN, LON_MIN]
    ys = [LAT_MIN, LAT_MIN, LAT_MAX, LAT_MAX, LAT_MIN]
    ax.plot(xs, ys, transform=ccrs.PlateCarree(),
            color='k', lw=0.9, zorder=30)


# ================== 单个子图 ==================
def plot_wind_panel(ax, u_mean, v_mean, title_text,
                    show_ticks=True, draw_branches=False):
    proj = ccrs.PlateCarree()
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=proj)

    ax.add_feature(cfeature.LAND.with_scale("50m"),
                   facecolor="#f2f1ee", zorder=1)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"),
                   facecolor="#daeeff", zorder=1)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"),
                   linewidth=0.35, edgecolor="#333333", zorder=5)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"),
                   linewidth=0.25, edgecolor="#777777",
                   linestyle=":", zorder=5)

    if show_ticks:
        ax.set_xticks(lon_ticks, crs=proj)
        ax.set_yticks(lat_ticks, crs=proj)
        ax.xaxis.set_major_formatter(LongitudeFormatter(number_format='.0f'))
        ax.yaxis.set_major_formatter(LatitudeFormatter(number_format='.0f'))
        ax.tick_params(labelsize=7.5, length=3, width=0.7, pad=2)
    else:
        ax.set_xticks([])
        ax.set_yticks([])

    draw_domain_border(ax)

    # 构建插值器
    u_interp, v_interp, u_s, v_s = build_interpolators(u_mean, v_mean)

    if draw_branches:
        bob_lons, bob_lats = trace_branch(u_interp, v_interp, BOB_SEEDS)
        scs_lons, scs_lats = trace_branch(u_interp, v_interp, SCS_SEEDS)

    # ---- 风矢量 ----
    lons   = u_s.lon.values
    lats_s = u_s.lat.values
    st     = QUIVER_STRIDE
    lq     = lons[::st]
    ltq    = lats_s[::st]
    uq     = u_s.values[::st, ::st].copy()
    vq     = v_s.values[::st, ::st].copy()

    spd_q = np.sqrt(uq**2 + vq**2)
    uq[spd_q < WIND_THRESHOLD] = np.nan
    vq[spd_q < WIND_THRESHOLD] = np.nan

    qv = ax.quiver(
        lq, ltq, uq, vq,
        transform=proj,
        scale=QUIVER_SCALE,
        scale_units="inches",
        width=0.0020,
        headwidth=4.0,
        headlength=4.5,
        headaxislength=3.8,
        color="#1a1a1a",
        alpha=0.82,
        zorder=20
    )

    if show_ticks:
        ax.quiverkey(
            qv,
            X=0.88, Y=-0.10,
            U=QUIVER_KEY_U,
            label=f"{QUIVER_KEY_U} m/s",
            labelpos="E",
            fontproperties={"size": 6.5},
            coordinates="axes",
            zorder=80
        )

    # ---- 关中轮廓（每张图都画，不影响比例）----
    draw_guanzhong_boundary(ax, proj=proj)

    # ---- 动态路径（仅最后一张图）----
    if draw_branches:
        draw_traced_branch(
            ax, bob_lons, bob_lats,
            color="#d62728", lw=1.80,
            label="BoB Branch",
            n_arrows=4, proj=proj
        )
        draw_traced_branch(
            ax, scs_lons, scs_lats,
            color="#e07b00", lw=1.60,
            label="SCS Branch",
            n_arrows=3, proj=proj
        )

    ax.set_title(title_text, fontsize=9, pad=4)


# ================== 图例与保存 ==================
def finalize_wind_layout(fig, outpath):
    plt.tight_layout(rect=[0.02, 0.07, 0.98, 0.95])

    legend_elements = [
        mpatches.Patch(
            facecolor="#ffe0a0", edgecolor="#8B0000",
            linewidth=1.2, alpha=0.85,
            label="Guanzhong Region"
        ),
        Line2D([0], [0], color="#d62728", lw=1.8,
               marker=">", markersize=4,
               label="BoB Southwesterly Branch (RK4 traced)"),
        Line2D([0], [0], color="#e07b00", lw=1.6,
               marker=">", markersize=4,
               label="SCS Southerly Branch (RK4 traced)"),
        Line2D([0], [0], color="#1a1a1a", lw=0,
               marker=r"$\rightarrow$", markersize=8,
               label="850 hPa Wind (length ~ speed)"),
    ]

    fig.legend(
        handles=legend_elements,
        loc="lower center",
        ncol=4,
        fontsize=7.5,
        frameon=True,
        framealpha=0.92,
        edgecolor="#aaaaaa",
        bbox_to_anchor=(0.48, 0.005),
        handlelength=2.5
    )

    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", outpath)


# ================== 主函数 ==================
def plot_wind_flat(outfile):
    fig = plt.figure(figsize=(10.8, 6.6))
    axs = [
        fig.add_subplot(2, 3, i + 1, projection=ccrs.PlateCarree())
        for i in range(6)
    ]

    for i, (start, end) in enumerate(date_ranges):
        print(f"Panel {i+1}/6: {start.strftime('%m-%d')} -> {end.strftime('%m-%d')}")
        u_mean, v_mean = load_wind_period(uwnd_ds, vwnd_ds, start, end)

        if u_mean.size == 0:
            axs[i].set_title("No Data", fontsize=9)
            continue

        plot_wind_panel(
            axs[i], u_mean, v_mean,
            f"{start:%m-%d} - {end:%m-%d}",
            show_ticks=(i == 0),
            draw_branches=(i == 5)
        )

    fig.suptitle(
        "850 hPa Wind Field - 2025 Spring (MAM, 6 Periods)\n"
        "Arrow length proportional to wind speed  |  "
        "Panel 6: dynamically traced pre-monsoon branches\n"
        "BoB: Bay of Bengal Southwesterly  |  "
        "SCS: South China Sea Southerly",
        fontsize=9.5, y=0.985
    )

    finalize_wind_layout(fig, outfile)


# ================== 入口 ==================
if __name__ == "__main__":
    print("Loading U-wind...")
    uwnd_ds = xr.open_dataset(
        r"E:\uwnd\uwnd_1980_2025_merged.nc",
        chunks={"time": 50}
    )
    print("Loading V-wind...")
    vwnd_ds = xr.open_dataset(
        r"E:\vwnd\vwnd_1980_2025_merged.nc",
        chunks={"time": 50}
    )

    outfile = os.path.join(
        output_dir,
        "850hPa_wind_2025_MAM_BoB_SCS_RK4_6panels.pdf"
    )
    plot_wind_flat(outfile)

    uwnd_ds.close()
    vwnd_ds.close()
    print("Done.")
